# ITRI626 Practical Project: Deep Learning for Image Classification
## Demonstration: Binary Dog Breed Classifier (Chihuahua vs Siberian Husky)

**Course**: ITRI626  
**Topic**: Dog Breed Classification  
**Purpose**: Proof-of-concept pipeline demonstrating data preparation, leakage-free splitting, augmentation, model training, evaluation metrics, and error analysis aligned with the ITRI626 assignment rubric.

### 1. Environment & Setup
First, we import the necessary libraries and set the random seed for full reproducibility.

In [ ]:
import os
import sys
import json
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

# Ensure project modules are discoverable
cwd = Path.cwd()
if str(cwd) not in sys.path:
    sys.path.append(str(cwd))

from dataset import get_dataloaders, DEFAULT_BREEDS, BREED_DISPLAY_NAMES
from models import get_model, BaselineCNN
from train import set_seed, plot_training_curves
from evaluate import evaluate_checkpoint

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')

### 2. Dataset Exploration & Reproducible Splits
We load the images from the Stanford Dogs dataset (`archive/images/Images`) and verify our 70% Train, 15% Validation, 15% Test split manifest.

In [ ]:
workspace_root = cwd.parent if cwd.name == 'Demo_2_breed_model' else cwd
train_loader, val_loader, test_loader, class_names = get_dataloaders(
    workspace_root=workspace_root,
    batch_size=16,
    seed=42
)

print(f'Classes: {class_names}')
print(f'Train batches: {len(train_loader)} ({len(train_loader.dataset)} images)')
print(f'Val batches:   {len(val_loader)} ({len(val_loader.dataset)} images)')
print(f'Test batches:  {len(test_loader)} ({len(test_loader.dataset)} images)')

### 3. Visualizing Sample Augmented Training Images
Let's inspect a batch of images after applying random rotations, horizontal flips, and color jitter.

In [ ]:
images, labels, paths = next(iter(train_loader))

# Denormalize for display
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    img = images[i].numpy().transpose((1, 2, 0))
    img = np.clip(std * img + mean, 0, 1)
    ax.imshow(img)
    ax.set_title(class_names[labels[i].item()], fontsize=11, fontweight='bold')
    ax.axis('off')
plt.suptitle('Sample Augmented Training Images', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

### 4. Comparing Model Architectures
We compare a custom 3-layer CNN baseline against pretrained transfer learning architectures (EfficientNetV2 and ResNet-18) matching ITRI626 rubric families.

In [ ]:
for arch in ['baseline', 'efficientnet_v2', 'resnet18', 'mobilenet_v3']:
    m = get_model(arch, num_classes=2, pretrained=True)
    total_p = sum(p.numel() for p in m.parameters())
    trainable_p = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'Model: {m.name:<28} | Total Params: {total_p:>10,} | Trainable: {trainable_p:>10,}')

### 5. Training History & Learning Curves
Load and display the training history and plot the loss and accuracy curves across epochs.

In [ ]:
history_path = Path('history.json') if Path('history.json').exists() else Path('Demo_2_breed_model/history.json')
if history_path.exists():
    with open(history_path, 'r', encoding='utf-8') as f:
        hist = json.load(f)
    epochs = range(1, len(hist['train_loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.plot(epochs, hist['train_loss'], 'o-', label='Train Loss')
    ax1.plot(epochs, hist['val_loss'], 's--', label='Val Loss')
    ax1.set_title('Cross Entropy Loss', fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.legend()
    ax1.grid(True, linestyle=':', alpha=0.6)

    ax2.plot(epochs, hist['train_acc'], 'o-', label='Train Accuracy')
    ax2.plot(epochs, hist['val_acc'], 's--', label='Val Accuracy')
    ax2.set_title('Accuracy (%)', fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.legend()
    ax2.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.show()
else:
    print('History file not found yet. Run train.py first.')

### 6. Held-Out Test Set Evaluation
Evaluate the best model checkpoint on the held-out test split, calculating Test Accuracy, Precision, Recall, and F1-Score.

In [ ]:
checkpoint_p = Path('best_model.pth') if Path('best_model.pth').exists() else Path('Demo_2_breed_model/best_model.pth')
if checkpoint_p.exists():
    metrics = evaluate_checkpoint(checkpoint_p, output_dir=checkpoint_p.parent)
else:
    print('Checkpoint not found. Run train.py to generate best_model.pth.')

### 7. Confusion Matrix & Error Analysis
Display the confusion matrix and visualize correct vs misclassified predictions to analyze error modes.

In [ ]:
cm_img_p = Path('confusion_matrix.png') if Path('confusion_matrix.png').exists() else Path('Demo_2_breed_model/confusion_matrix.png')
pred_img_p = Path('sample_predictions.png') if Path('sample_predictions.png').exists() else Path('Demo_2_breed_model/sample_predictions.png')

if cm_img_p.exists():
    plt.figure(figsize=(6, 5))
    plt.imshow(Image.open(cm_img_p))
    plt.axis('off')
    plt.title('Test Set Confusion Matrix', fontweight='bold')
    plt.show()

if pred_img_p.exists():
    plt.figure(figsize=(10, 8))
    plt.imshow(Image.open(pred_img_p))
    plt.axis('off')
    plt.title('Sample Test Predictions (Green: Correct, Red: Error)', fontweight='bold')
    plt.show()

### 8. Scaling to Group Models
To scale this demo to the individual project requirements:
1. **`JP_Model/`**: Adapt for Architecture Family 1 (e.g., Deep Baseline CNN / ResNet).
2. **`Lindani_Model/`**: Adapt for Architecture Family 2 (e.g., DenseNet / EfficientNet / MobileNet).
3. **`Sulaiman_Model/`**: Adapt for Architecture Family 3 (e.g., Vision Transformer ViT or Swin Transformer).
4. Use the same shared data split across all three models for a fair comparison!